# A Game of Thrones — AlphaZero training on Google Colab

This notebook clones the repository, installs the wheel, runs a smoke gate and
then launches a **resumable** run into `run6`. Checkpoints are written to Google
Drive, so a runtime that disconnects can continue from the last iteration.

Leaves are scored by the quantized ONNX engine (`onnx-int8`), which measured
2.35x the PyTorch path end to end and 2.46x on the training loop; a preflight
cell builds the graph, checks the policy KL against torch and fails loudly if
the installed wheel would silently fall back to the slow engine.

`run3` predates the sequential Westeros resolution and the PIMC deck resampling;
it is a pre-change baseline, not something to resume. The wheel is cached on
Drive by source tree + platform + Python, so a docs-only commit reuses it while a
Rust/Python/data change rebuilds it.

**Before you start**

1. `Runtime > Change runtime type`: **any runtime works**. With a GPU (the
   free tier offers the T4) the gradient pass runs on it; on a CPU runtime the
   whole pipeline still runs, just slower. Nothing here requires CUDA, so a run
   can be split between GPU and CPU sessions.
2. The GitHub repository is **private**. Create a fine-grained read-only token
   for it and add it in the 🔑 panel (left sidebar) as a secret named
   `GITHUB_TOKEN`. If the repository is public, skip this.
3. Keep this tab open while training runs. If Colab disconnects you, the next
   session resumes from the last checkpoint on Drive.


In [ ]:
import os
import sys

import torch

affinity = len(os.sched_getaffinity(0)) if hasattr(os, "sched_getaffinity") else None
print("python:", sys.version.split()[0])
print("torch:", torch.__version__, "| cuda available:", torch.cuda.is_available())
print("usable cpus:", affinity or os.cpu_count())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
else:
    print("gpu: none (CPU runtime; training will run on the CPU)")

In [ ]:
%cd /content
import os
import subprocess
from pathlib import Path

REPO_DIR = Path("/content/game-of-thrones")
if not REPO_DIR.exists():
    try:
        from google.colab import userdata

        token = userdata.get("GITHUB_TOKEN")
    except Exception:
        token = None
    url = "https://github.com/canditoo2/game-of-thrones.git"
    if token:
        url = f"https://{token}@github.com/canditoo2/game-of-thrones.git"
    subprocess.run(
        ["git", "clone", "--depth", "1", url, str(REPO_DIR)],
        check=True,
    )

# A failed pull must stop the notebook: otherwise Colab would install the old
# commit's cached wheel and run6 would train stale code while looking healthy.
subprocess.run(
    ["git", "-C", str(REPO_DIR), "fetch", "--depth", "1", "origin", "main"],
    check=True,
)
subprocess.run(
    ["git", "-C", str(REPO_DIR), "reset", "--hard", "FETCH_HEAD"],
    check=True,
)
head = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True
).strip()
fetched = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "FETCH_HEAD"], text=True
).strip()
assert head == fetched, (head, fetched)
subject = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "log", "-1", "--format=%s"], text=True
).strip()
print("training commit", head[:12], "|", subject)
os.chdir(REPO_DIR)


## 1. Mount Drive and install the engine

The engine is a Rust extension. The wheel cache key is the **source tree**
(`rust/`, `src/`, `pyproject.toml`) plus platform and Python, so changing only
documentation does not rebuild it. `RUN_DIR` is `run6`; `run3` is obsolete.


In [ ]:
from google.colab import drive

drive.mount("/content/drive")

import glob
import hashlib
import json
import os
import platform
import subprocess
import sys
import zipfile
from pathlib import Path

REPO_DIR = Path("/content/game-of-thrones")
RUN_ROOT = Path("/content/drive/MyDrive/agot")
RUN_DIR = RUN_ROOT / "run6"
ITERATIONS = 300  # total across sessions; resume picks up from the state file
RUN_DIR.mkdir(parents=True, exist_ok=True)

commit = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True
).strip()
source_tree = subprocess.check_output(
    [
        "git",
        "-C",
        str(REPO_DIR),
        "rev-parse",
        "HEAD:rust",
        "HEAD:src",
        "HEAD:pyproject.toml",
    ],
    text=True,
).strip()
source_key = hashlib.sha1(source_tree.encode()).hexdigest()[:12]
wheel_key = (
    f"{source_key}-{sys.platform}-{platform.machine()}-"
    f"{sys.version_info.major}{sys.version_info.minor}"
)
wheel_dir = RUN_ROOT / "wheels" / wheel_key
wheel_dir.mkdir(parents=True, exist_ok=True)


def usable_wheels():
    paths = sorted(glob.glob(str(wheel_dir / "agot-*.whl")))
    valid = [path for path in paths if zipfile.is_zipfile(path)]
    if sys.platform.startswith("linux"):
        tags = ("manylinux", "linux")
    elif sys.platform == "win32":
        tags = ("win",)
    elif sys.platform == "darwin":
        tags = ("macosx",)
    else:
        tags = (sys.platform,)
    return [
        path
        for path in valid
        if any(tag in os.path.basename(path) for tag in tags)
    ]


wheels = usable_wheels()
if wheels:
    print("using cached wheel", wheels[0])
else:
    print("building the Rust wheel for", wheel_key)
    !curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y
    os.environ["PATH"] = (
        os.path.expanduser("~/.cargo/bin") + os.pathsep + os.environ["PATH"]
    )
    os.environ["MATURIN_PEP517_ARGS"] = "--profile release"
    os.chdir(REPO_DIR)
    !pip wheel . --no-deps --wheel-dir {wheel_dir}
    wheels = usable_wheels()

assert wheels, "the wheel build produced no usable artefact"
wheel = Path(wheels[0])
with zipfile.ZipFile(wheel) as archive:
    names = archive.namelist()
assert any(name.startswith("agot/_core") for name in names), names[:10]
assert "agot/data/map_base_6p.json" in names
!pip install -q --force-reinstall --no-deps {wheel}
!pip install -q pytest pytest-cov ruff
!pip install -q "onnx>=1.15" "onnxruntime>=1.17"

import agot
from agot import _core

installed = Path(agot.__file__).resolve()
assert str(REPO_DIR) not in str(installed), installed
print("agot", agot.__version__, "| installed", installed)
print("engine", _core.backend())
assert _core.backend() == "rust"

manifest = {
    "commit": commit,
    "source_key": source_key,
    "wheel": wheel.name,
    "python": sys.version,
    "platform": sys.platform,
    "machine": platform.machine(),
    "iterations": ITERATIONS,
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
}
(RUN_DIR / "run.json").write_text(
    json.dumps(manifest, indent=2), encoding="utf-8"
)
print("checkpoints ->", RUN_DIR)


In [ ]:
import subprocess

import agot
from agot import _core
from agot.engine.game import BACKEND, Game, GameConfig
from agot.engine.rust import RustGame

config = GameConfig(seed=0, use_westeros_decks=False)
assert config.sequential_westeros is True
assert "pending_choice" in dir(_core.Game)
game = Game.new(config)
assert isinstance(game, RustGame)
assert _core.backend() == "rust"
train_help = subprocess.check_output(["agot", "train", "--help"], text=True)
assert "--device" in train_help, (
    "the installed wheel predates --device: delete the cached wheel and re-run "
    "the install cell"
)
assert game.legal_actions()
print(
    f"agot {agot.__version__} | package engine {BACKEND} | "
    f"extension {_core.backend()} | legal actions {len(game.legal_actions())}"
)

!agot demo --policy greedy --seed 1
!python -m pytest tests/test_map.py tests/test_engine.py tests/test_rust_engine.py tests/test_batched_selfplay.py -q


## 1b. Preflight: the int8 leaf engine

`run6` trains with `--leaf-backend onnx-int8` (the CLI default for `agot
train` since the speed work). The engine is worth 2.4x the PyTorch path, so
a silent fallback would cost hours before anyone noticed. This cell builds
the graph, scores real leaf batches through it and checks the policy KL
against torch stays inside the budget the test suite enforces.


In [ ]:
# Preflight: prove the leaf engine this run trains on actually works here.
# A stale wheel would silently fall back to torch (2.4x slower) and look like a
# slow run rather than a broken install, so fail here instead.
import time

import numpy as np
import torch

from agot.nn.model import PolicyValueNet
from agot.nn.onnx_backend import LeafBackend
from agot.training import rust_engine

assert rust_engine.resolve_leaf_backend("auto", training=True) == "onnx-int8", (
    "the installed wheel predates the int8 training default: delete the cached "
    "wheel and re-run the install cell"
)

model = PolicyValueNet(hidden=256, value_context=True).eval()
model.save("checkpoints/preflight.pt")
backend = LeafBackend.from_model(
    model, "onnx-int8", checkpoint="checkpoints/preflight.pt", threads=1
)

rng = np.random.default_rng(0)
kls = []
costs = {"torch": [], "onnx-int8": []}
for _ in range(20):
    leaves = int(rng.integers(1, 9))
    cand = int(rng.integers(5, 30))
    state = rng.random((leaves, 97), dtype=np.float32)
    actions = rng.random((leaves * cand, 141), dtype=np.float32)
    index = np.repeat(np.arange(leaves, dtype=np.int64), cand)
    with torch.inference_mode():
        start = time.perf_counter()
        logits, values = model.forward_packed(
            torch.from_numpy(state),
            torch.from_numpy(actions),
            torch.from_numpy(index),
        )
        costs["torch"].append(time.perf_counter() - start)
    start = time.perf_counter()
    q_logits, q_values = backend(state, actions, index)
    costs["onnx-int8"].append(time.perf_counter() - start)
    assert np.isfinite(q_logits).all() and np.isfinite(q_values).all()
    p = np.exp(logits.numpy() - logits.numpy().max())
    p /= p.sum()
    q = np.exp(q_logits - q_logits.max())
    q /= q.sum()
    kls.append(float(np.sum(p * np.log((p + 1e-12) / (q + 1e-12)))))

mean_kl = float(np.mean(kls))
assert mean_kl < 1e-3, f"int8 mean KL {mean_kl:.2e} nats is outside the budget"
torch_ms = 1e3 * float(np.median(costs["torch"]))
int8_ms = 1e3 * float(np.median(costs["onnx-int8"]))
print(f"int8 leaf KL mean {mean_kl:.2e} nats (budget 1e-3)")
print(f"per-call cost: torch {torch_ms:.2f} ms, int8 {int8_ms:.2f} ms ({torch_ms / int8_ms:.1f}x)")


## 2. Smoke gate (a few minutes)

Three short iterations on six games. This checks the real sequential/PIMC
pipeline, the native driver, checkpoint reload and the log — not just that the
process starts. A handful of iterations will not prove learning; a crash, a
frozen loss or a stale wheel will show up here instead of hours into a run.


In [ ]:
import shutil
import subprocess

shutil.rmtree("checkpoints/smoke", ignore_errors=True)
result = subprocess.run(
    [
        "agot",
        "train",
        "--iterations",
        "3",
        "--games",
        "6",
        "--workers",
        "2",
        "--batch-games",
        "3",
        "--train-threads",
        "2",
        "--buffer",
        "20000",
        "--eval-games",
        "6",
        "--eval-every",
        "3",
        "--leaf-backend",
        "onnx-int8",
        "--device",
        "auto",
        "--checkpoint",
        "checkpoints/smoke/agot.pt",
    ],
    capture_output=True,
    text=True,
)
print(result.stdout)
print(result.stderr)
if result.returncode:
    raise RuntimeError(
        f"agot train exited {result.returncode}\n"
        f"--- last stderr ---\n{result.stderr[-6000:]}"
    )



In [ ]:
from pathlib import Path

from agot.nn.model import PolicyValueNet
from agot.training.checkpoint import RunState, state_path
from agot.training.report import load_records

log = Path("checkpoints/smoke/agot.log.jsonl")
records = load_records(log)
assert len(records) == 3, records
for record in records:
    print(
        f"iter {record['iteration']:>3}  "
        f"policy_loss {record['policy_loss']:.4f}  "
        f"value_loss {record['value_loss']:.4f}  "
        f"({record.get('seconds', 0):.0f}s)"
    )
    assert record.get("policy_loss") is not None
    assert record.get("leaf_backend") == "onnx-int8", record.get("leaf_backend")
    assert record.get("value_loss") is not None

losses = [record["policy_loss"] for record in records]
assert len(set(losses)) > 1, "policy loss never changed across three iterations"
assert losses[-1] < 4.0, losses[-1]
PolicyValueNet.load("checkpoints/smoke/agot.pt")
state = RunState.load(state_path("checkpoints/smoke/agot.pt"))
assert state is not None
assert state.iteration == 3
print("OK: the pipeline ran, the loss moved and the checkpoint reloads.")


## 2b. Optional warm start

`run5` reached 105 iterations and is the warm-start candidate. The cell
below is a no-op unless you set `WARM_START_FROM`; it refuses to overwrite
an existing `run6` and refuses to seed a run whose `--gate-mode` the launch
cell would change.


In [ ]:
# Optional warm start. run5 reached 105 iterations and is the documented
# warm-start candidate; a fresh start is equally valid and is what the run6
# configuration was tuned for. Leave WARM_START_FROM as None for a fresh run6.
#
# The CLI refuses to resume a run with a different --gate-mode than the one in
# its state file, so a run5 trained inline cannot be resumed by the deferred
# launch below. This cell prints the source's mode and refuses to guess.
import shutil

from agot.training.checkpoint import RunState, state_path

WARM_START_FROM = None  # e.g. "run5"

if WARM_START_FROM is None:
    print("no warm start: run6 starts fresh")
else:
    source = RUN_ROOT / WARM_START_FROM
    target = RUN_DIR
    if (target / "agot.pt").exists():
        print(f"{target} already has a checkpoint; refusing to overwrite")
    elif not (source / "agot.pt").exists():
        print(f"no checkpoint in {source}")
    else:
        source_state = RunState.load(state_path(source / "agot.pt"))
        print(
            f"{WARM_START_FROM}: iteration {source_state.iteration}, "
            f"gate_mode {source_state.gate_mode!r}"
        )
        if source_state.gate_mode not in (None, "deferred"):
            print(
                "that run was gated inline; the launch cell below uses deferred, "
                "and the CLI refuses to switch modes on resume. Set "
                "--gate-mode inline there (and edit the cell) before seeding."
            )
        else:
            for name in ("agot.pt", "agot.state.pt", "agot.log.jsonl"):
                origin = source / name
                if origin.exists():
                    shutil.copy2(origin, target / name)
                    print("copied", origin.name)
            print("run6 seeded from", source)


## 3. Launch the training run

`run6` starts fresh (or from the optional warm start above); `run3` is a
pre-change baseline. The target is **60
iterations** of 48 games, spread across sessions: a free 2-vCPU runtime usually
takes roughly 5?15 minutes per iteration depending on the batch and eval, so
re-run this cell after a disconnect and it resumes from `agot.state.pt`.

Every 10 iterations the run logs two checks:

- a cheap 12-game **anchor** against random and greedy, for continuity with the
  old runs;
- a **relative gate**: 60 games of the current net (3 seats) against
  `agot-best.pt` (3 seats), swapped each game, with per-house win rates. A new
  checkpoint replaces the best only if it clears the overall threshold and every
  house matches or beats the previous best in that seat. The optional
  `--gate-min-house` adds an absolute floor. After three consecutive gate
  failures the run stops, leaving the gate-approved best in place.

Watch the eval/gate trend and use the gate-approved `agot-best.pt`, not the
last `agot.pt`.


In [ ]:
import os

import subprocess

from agot.nn.model import PolicyValueNet
from agot.training.checkpoint import RunState, log_path, state_path
from agot.training.report import load_records

checkpoint = RUN_DIR / "agot.pt"
state_file = state_path(checkpoint)
log_file = log_path(checkpoint)

resume = checkpoint.exists() and state_file.exists()
if resume:
    try:
        PolicyValueNet.load(str(checkpoint))
    except Exception as exc:  # old head or a truncated torch.save
        print("checkpoint unusable, starting fresh:", exc)
        resume = False
if resume:
    done = RunState.load(state_file).iteration
    logged = len(load_records(log_file))
    if logged < done:
        print(
            f"checkpoint says iter {done} but the log has {logged} records: "
            "the previous launch died before training, starting fresh"
        )
        resume = False

if not resume:
    for stale in (checkpoint, state_file, log_file, RUN_DIR / "agot-best.pt"):
        if stale.exists():
            stale.unlink()
            print("removed stale", stale.name)

state = RunState.load(state_file) if resume else None
done = state.iteration if state is not None else 0
print("checkpoint:", checkpoint)
print(("resuming at iteration " if state is not None else "starting fresh "), done)

if done >= ITERATIONS:
    print(f"already at {done}/{ITERATIONS}; raise ITERATIONS to continue")
else:
    command = [
        "agot",
        "train",
        "--iterations",
        str(ITERATIONS),
        "--simulations",
        "128",
        "--determinizations",
        "1",
        "--phase-budgets",
        "default",
        "--leaf-backend",
        "onnx-int8",
        "--games",
        "64",
        "--workers",
        "2",
        "--batch-games",
        "16",
        "--buffer",
        "80000",
        "--opponent-rate",
        "0.2",
        "--buffer-save-every",
        "1",
        "--train-threads",
        "2",
        "--device",
        "auto",
        "--temperature-rounds",
        "3",
        "--final-temperature",
        "0.25",
        "--eval-games",
        "40",
        "--eval-every",
        "25",
        "--gate-games",
        "120",
        "--gate-mode",
        "deferred",
        "--gate-threshold",
        "0.55",
        "--gate-min-house",
        "0.0",
        "--gate-house-tolerance",
        "0.25",
        "--gate-house-margin",
        "0.0",
        "--gate-patience",
        "4",
        "--checkpoint",
        str(checkpoint),
    ]
    if state is not None:
        command += ["--resume", str(checkpoint)]
    # Colab: a child's stdout on fd 1 does not reach the cell, so stream it.
    env = dict(os.environ, OMP_NUM_THREADS="1", MKL_NUM_THREADS="1")
    process = subprocess.Popen(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=env,
    )
    tail: list[str] = []
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="", flush=True)
        tail.append(line)
        if len(tail) > 200:
            tail.pop(0)
    code = process.wait()
    if code:
        raise RuntimeError(
            f"agot train exited {code}\n--- last output ---\n{''.join(tail[-40:])}"
        )


In [ ]:
# Play the queued gate candidates in the background (second tab, or after
# a reconnect). Chunks are atomic, so an interruption resumes here.
!agot gate --checkpoint $RUN_DIR/agot.pt --max-minutes 90


## 4. Monitor

The training cell prints one line per iteration. This cell reads the JSONL
tolerantly (a disconnect can truncate the last line), prints the last few
iterations and renders the loss/eval curves to `/content/run-report.html`.


In [ ]:
from pathlib import Path

from IPython.display import HTML, display

from agot.training.report import load_records, render_file

log = RUN_DIR / "agot.log.jsonl"
if not log.exists():
    print("no log yet — run the training cell first")
else:
    records = load_records(log)
    for record in records[-5:]:
        rate = record.get("eval")
        eval_text = f"  eval {rate:.3f}" if rate is not None else ""
        gate = record.get("gate")
        gate_text = ""
        if gate is not None:
            gate_text = f"  gate {gate:.3f}"
            if record.get("gate_passed") is False:
                gate_text += " (kept best)"
            elif record.get("gate_passed") is True:
                gate_text += " (new best)"
        print(
            f"iter {record['iteration']:>4}  "
            f"policy_loss {record['policy_loss']:.4f}  "
            f"value_loss {record['value_loss']:.4f}{eval_text}{gate_text}"
        )
    page = render_file(
        log,
        out="/content/run-report.html",
        refresh=0,
        target_iterations=ITERATIONS,
    )
    display(HTML(Path(page).read_text(encoding="utf-8")))


## 5. Evaluate the best checkpoint

`agot gauntlet` puts the saved network in one rotating seat against five
opponents. The explicit `--simulations 128 --determinizations 2` matters: the
CLI's cheap default is 24/1, which would measure a weaker search than the run
trained with. The cells use `agot-best.pt` when it exists and fall back to the
last checkpoint before the first evaluation.


In [ ]:
from pathlib import Path

best = RUN_DIR / "agot-best.pt"
checkpoint = best if best.exists() else RUN_DIR / "agot.pt"
print("evaluating", checkpoint)
!agot gauntlet --candidate net:{checkpoint} --opponents random --games 60 --simulations 128 --determinizations 2


In [ ]:
from pathlib import Path

best = RUN_DIR / "agot-best.pt"
first = best if best.exists() else RUN_DIR / "agot.pt"
!agot round-robin --checkpoints {first} {RUN_DIR}/agot.pt --games 30 --simulations 128 --determinizations 2


## Troubleshooting

- **Out of memory** (a free runtime has ~12 GB): the launch uses an 80k replay
  buffer. Lower it to `--buffer 40000` and/or `--games 32`; the batch is 12
  games per worker, so a smaller `--batch-games` also lowers peak MCTS memory.
- **Disconnected**: re-run the clone/install cell and the launch cell; it
  installs the cached wheel and resumes `run6` from the Drive checkpoint. The
  replay buffer restarts empty by design, so a transient loss spike on the first
  resumed iteration is expected.
- **`leaf_backend: torch` in `agot.log.jsonl`**: the `[onnx]` extra is
  missing, so `auto` fell back to PyTorch and the run is ~2.4x slower. Re-run
  the install cell (it installs `onnx`/`onnxruntime`) and restart the run.
- **`agot.onnx-int8.onnx` on Drive**: the quantized graph, rebuilt once per
  iteration beside the checkpoint (~0.6 MB). It is a cache, not a checkpoint:
  deleting it only costs one rebuild.
- **Stale wheel**: the cache key covers `rust/`, `src/` and `pyproject.toml`;
  `run6/run.json` records the commit and wheel name. If in doubt, delete that
  `wheels/<key>` folder and re-run the install cell.
- **Policy loss flat for many iterations**: read `docs/methods.md` and compare
  the eval/gate trend. The old absolute ~2.0 signature was measured with the
  smaller choice action set; with generic choices, use the trend.
- **Gate stops the run**: three consecutive losses to the previous best leave
  the best in place and stop the session; the last checkpoint is still
  `agot.pt`. Resume later (or raise `--gate-patience`) if you want to keep
  exploring. Lower `--gate-min-house` only if the per-house requirement is too
  strict for the number of games.
- **`agot: command not found`**: re-run the install cell.
- **You pulled a new commit in the same session**: the extension is compiled
  code and cannot be reloaded in place. Run the install cell, then
  `Runtime > Restart runtime`.
- **More throughput**: keep `--workers 2` and `--batch-games 12` on a 2-vCPU
  runtime; the printed `seconds` per iteration is the number to watch.
